# Analyse extraction quality

0. Extract from manual anotation args in process format
1. Builds optimal pairs with sentence-embedding cosine similarity (fast)
2. **Metric vs. Manual depuration** (overall)
    - Precision = TP / Predicted Positives
    
    Where: True Positives (TP) = total_args_kept for each model., Predicted Positives = total_args for each model.

3. **Metric vs. Manual annotation** (overall and by goal):
    - rougeL_f1_avg (Measures recall/precision of overlapping n-grams or longest common subsequence, pairwise with pairs if max similarity grater than 0.75 is a positive and avg.)
    - bleu_avg (Precision-oriented metric on n-grams, pairwise with pairs of max similarity and avg.)
    - Recall@k: proportion of gold arguments retrieved.
    - Precision: proportion of retrieved arguments that match gold.
    - F1-score: balance of precision and recall.
    - Coverage Ratio: (# of distinct reference arguments covered) ÷ (total reference arguments).
    - Overgeneration: (# retrieved – # matched) ÷ (# retrieved).

4.  **Metric vs. Manual annotation** (between models agreement): 
    - Fleiss’ κ across models (How consistently multiple raters assign labels to the same units, so in extraction case since models don’t label the exact same argument units, agreement is mostly chance-level.)
    - Jaccard (Overlap-based set similarity) 
    - Fuzzy-matching Jaccard (cosine/embedding-based instead of exact string)
    - % of gold arguments recovered by ≥1, 2, 3 and all model.
    - % of model arguments supported by another model.




In [ ]:
#%pip install -q sentence-transformers rouge-score bert-score nltk hf_xet


Note: you may need to restart the kernel to use updated packages.


In [4]:
import nltk
nltk.download('punkt', quiet=True)

import re
from nltk.tokenize import TreebankWordTokenizer
_tok = TreebankWordTokenizer()

import os, re, json, math
from typing import List, Dict, Tuple
import numpy as np
import pandas as pd
from collections import defaultdict, OrderedDict

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from scipy.optimize import linear_sum_assignment
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

# BERTScore (computationally expensive)
USE_BERTSCORE = False
if USE_BERTSCORE:
    from bert_score import score as bert_score

### 0. Extract from manual anotation args in process format

In [5]:
import os, re, json
import pandas as pd
from collections import defaultdict

def make_annotations_outputs(
    path_annotations,
    prefix,
    excel_name="TFM-ExtractedArgumentsAnnotations.xlsx",
    json_basename=None,
    txt_basename=None,
    keep_duplicates=True
):
    # 1. Load Excel and normalize column names
    excel_path = os.path.join(path_annotations, excel_name)
    if not os.path.exists(excel_path):
        raise FileNotFoundError(f"Excel not found at: {excel_path}")
    df = pd.read_excel(excel_path)
    df.rename(columns={c: c.strip().lower() for c in df.columns}, inplace=True)

    def find_col(pattern):
        for c in df.columns:
            if re.search(pattern, c, flags=re.I):
                return c
        raise KeyError(f"No column matches pattern: {pattern}")

    col_prefix = find_col(r"\bprefix\b")
    col_page   = find_col(r"\bpage\b")
    col_ods    = find_col(r"\bods")
    col_arg    = find_col(r"\bargument")

    # Filter by prefix
    msk = df[col_prefix].astype(str).str.startswith(prefix)
    df = df[msk].copy()
    if df.empty:
        raise ValueError(f"No rows in Excel match prefix '{prefix}'.")

    #2. Parse ODS list per row
    def parse_ods_list(x):
        if pd.isna(x):
            return []
        parts = re.split(r"[,\s]+", str(x).strip())
        out = []
        for p in parts:
            if p == "":
                continue
            try:
                out.append(int(p))
            except:
                pass
        return out

    #3. Build grouped dict {(ods, page): [arguments]} and flat list of all arguments
    grouped = defaultdict(list)
    grouped_by_goal = defaultdict(list)
    all_args = []

    for _, row in df.iterrows():
        try:
            page = int(pd.to_numeric(row[col_page], errors="coerce"))
        except Exception:
            continue
        if pd.isna(page):
            continue

        arg = str(row[col_arg]).strip()
        if not arg or arg.lower() == "nan":
            continue

        ods_list = parse_ods_list(row[col_ods])
        all_args.append(arg)
        for ods in ods_list:
            key = (int(ods), int(page))
            if keep_duplicates or (arg not in grouped[key]):
                grouped[key].append(arg)
            if keep_duplicates or (arg not in grouped_by_goal[int(ods)]):
                grouped_by_goal[int(ods)].append(arg)

    #4. Output files (.json per ODS and page, .txt for all arguments)
    records = []
    for (ods, page), args in sorted(grouped.items(), key=lambda x: (x[0][0], x[0][1])):
        records.append({
            "ods": int(ods),
            "page": int(page),
            "arguments": args,
            "source_file": f"{prefix}annotations.xlsx",
        })

    if json_basename is None:
        json_basename = f"{prefix}_AllArgs_annotations"
    if txt_basename is None:
        txt_basename = f"{prefix}_AllArgs_annotations_arguments"
    json_path = os.path.join(path_annotations, f"{json_basename}.json")
    txt_path  = os.path.join(path_annotations, f"{txt_basename}.txt")
    txt_bygoal_path = os.path.join(path_annotations, f"{prefix}_AllArgs_annotations_by_goal.txt")

    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(records, f, indent=2, ensure_ascii=False)
    with open(txt_path, "w", encoding="utf-8") as f:
        for a in all_args:
            cleaned_a = str(a).replace('\n', ' ').strip()
            f.write(f"'{cleaned_a}',\n")
    with open(txt_bygoal_path, "w", encoding="utf-8") as f:
        json.dump(grouped_by_goal, f, indent=2, ensure_ascii=False)

    results = {"json_path": json_path, "txt_path": txt_path, "records_count": len(records), "args_count": len(all_args)}
    print(", ".join(f"{k}: {v}" for k, v in results.items()))
    return results


In [6]:
path_annotations  = "..\\Data\\Annotations\\"
prefix = "GLOBAL_SGD2023_"

make_annotations_outputs(
    path_annotations,
    prefix,
    excel_name= "TFM-ExtractedArgumentsAnnotations.xlsx",
    json_basename= None,
    txt_basename= None,
    keep_duplicates = True)

prefix = "GLOBAL_SGD2024_"

make_annotations_outputs(
    path_annotations,
    prefix,
    excel_name= "TFM-ExtractedArgumentsAnnotations.xlsx",
    json_basename= None,
    txt_basename= None,
    keep_duplicates = True)

prefix = "GLOBAL_SGD2025_"

make_annotations_outputs(
    path_annotations,
    prefix,
    excel_name= "TFM-ExtractedArgumentsAnnotations.xlsx",
    json_basename= None,
    txt_basename= None,
    keep_duplicates = True)

prefix = "OCDE_2024_"

make_annotations_outputs(
    path_annotations,
    prefix,
    excel_name= "TFM-ExtractedArgumentsAnnotations.xlsx",
    json_basename= None,
    txt_basename= None,
    keep_duplicates = True)

prefix = "G20_2024_"

make_annotations_outputs(
    path_annotations,
    prefix,
    excel_name= "TFM-ExtractedArgumentsAnnotations.xlsx",
    json_basename= None,
    txt_basename= None,
    keep_duplicates = True)

json_path: ..\Data\Annotations\GLOBAL_SGD2023__AllArgs_annotations.json, txt_path: ..\Data\Annotations\GLOBAL_SGD2023__AllArgs_annotations_arguments.txt, records_count: 229, args_count: 235
json_path: ..\Data\Annotations\GLOBAL_SGD2024__AllArgs_annotations.json, txt_path: ..\Data\Annotations\GLOBAL_SGD2024__AllArgs_annotations_arguments.txt, records_count: 198, args_count: 415
json_path: ..\Data\Annotations\GLOBAL_SGD2025__AllArgs_annotations.json, txt_path: ..\Data\Annotations\GLOBAL_SGD2025__AllArgs_annotations_arguments.txt, records_count: 149, args_count: 166
json_path: ..\Data\Annotations\OCDE_2024__AllArgs_annotations.json, txt_path: ..\Data\Annotations\OCDE_2024__AllArgs_annotations_arguments.txt, records_count: 151, args_count: 150
json_path: ..\Data\Annotations\G20_2024__AllArgs_annotations.json, txt_path: ..\Data\Annotations\G20_2024__AllArgs_annotations_arguments.txt, records_count: 51, args_count: 50


{'json_path': '..\\Data\\Annotations\\G20_2024__AllArgs_annotations.json',
 'txt_path': '..\\Data\\Annotations\\G20_2024__AllArgs_annotations_arguments.txt',
 'records_count': 51,
 'args_count': 50}

### 1. Define functions for Metric vs. Manual depuration

Definitions:
- Ground truth positives (gold) = total_args of the annotation row.
- True Positives (TP) = total_args_kept for each model.
- Predicted Positives = total_args for each model.

Metrics:
1. **Precision** = TP / Predicted Positives

In [12]:
def compute_prf(path_metrics_extracted, filename="AllArgs_overall_summary.csv"):
    path = os.path.join(path_metrics_extracted, filename)
    df = pd.read_csv(path)

    rows = []
    for _, row in df.iterrows():
        model = row["model"]
        document = row["prefix"]
        if model == "annotation":
            gold_total = row["total_args"]
            continue

        tp = int(row["total_args_kept"])
        pred_total = int(row["total_args"])

        precision = tp / pred_total if pred_total > 0 else 0
        rows.append({
            "document": document,
            "model": model,
            "total": gold_total,
            "extracted_total": pred_total,
            "real_argument": tp,
            "precision": round(precision, 4),
        })

    results = pd.DataFrame(rows)
    return results

### 2. Define functions for Metric vs. Manual annotation


In [8]:
# Load argument texts and goal-based JSON

def load_args_txt(path):
    args = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            s = line.strip().rstrip(",")
            s = re.sub(r"^'+|^\"+|'+$|\"+$", "", s).strip()
            if s:
                args.append(s)
    return args

def load_by_goal_json_txt(path):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    return {str(k): v for k, v in data.items()}

def build_path_overall(path_dir, prefix, model_name):
    fn = f"{prefix}__AllArgs_{model_name}_arguments.txt"
    return os.path.join(path_dir, fn)

In [9]:
#### Text + embeddings

_EMBED = None
def get_embedder(model_name: str = "sentence-transformers/all-MiniLM-L6-v2"):
    global _EMBED
    if _EMBED is None:
        _EMBED = SentenceTransformer(model_name)
    return _EMBED

def normalize(s):
    s = re.sub(r"\s+", " ", s.strip())
    return s

def embed_texts(texts):
    emb = get_embedder().encode([normalize(t) for t in texts], batch_size=64, show_progress_bar=False, convert_to_numpy=True, normalize_embeddings=True)
    return emb

# Matching by similarity argument extracted with ground truth (minimizing Hungarian cost = 1 - cosine similarity)
def optimal_matching(refs, preds):

    if len(refs) == 0 or len(preds) == 0:
        return [], np.zeros((len(refs), len(preds)))
    E_ref = embed_texts(refs)
    E_pred = embed_texts(preds)
    S = cosine_similarity(E_ref, E_pred) 

    # Optimal search (min cost)
    cost = 1.0 - S
    r_idx, p_idx = linear_sum_assignment(cost)
    pairs = [(ri, pj, float(S[ri, pj])) for ri, pj in zip(r_idx, p_idx)]

    return pairs, S

# Pairwise metrics
_rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
smooth = SmoothingFunction().method3

def rougeL_f1(a, b):
    return _rouge.score(normalize(a), normalize(b))["rougeL"].fmeasure

def _tok_words(s: str):
    return _tok.tokenize(re.sub(r"\s+", " ", s.strip()))

def bleu_pair(a: str, b: str) -> float:
    ref = [_tok_words(a)]
    hyp = _tok_words(b)
    from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
    smooth = SmoothingFunction().method3
    return sentence_bleu(ref, hyp, smoothing_function=smooth,
                         weights=(0.25, 0.25, 0.25, 0.25))


def berts_pairwise(a_list, b_list):
    P, R, F = bert_score(a_list, b_list, lang="en", rescale_with_baseline=True)
    return float(F.mean())

###################################
# Scoring one model vs ground truth

def evaluate_one(pred_args, gold_args, sim_threshold = 0.75,
                 use_bertscore = USE_BERTSCORE):
    
    pairs, S = optimal_matching(gold_args, pred_args) 

    # Best alignment stats
    sims = [s for _, _, s in pairs]
    avg_sim = float(np.mean(sims)) if sims else 0.0

    # Classification by similarity threshold
    matched = sum(s >= sim_threshold for s in sims) if sims else 0
    precision = matched / max(len(pred_args), 1)
    recall    = matched / max(len(gold_args), 1)
    f1        = 2*precision*recall / max((precision+recall), 1e-9)

    # ROUGE-L / BLEU over aligned pairs (macro average)
    rouge_vals, bleu_vals = [], []
    for gi, pj, _ in pairs:
        rouge_vals.append(rougeL_f1(gold_args[gi], pred_args[pj]))
        bleu_vals.append(bleu_pair(gold_args[gi], pred_args[pj]))
    rougeL_avg = float(np.mean(rouge_vals)) if rouge_vals else 0.0
    bleu_avg   = float(np.mean(bleu_vals)) if bleu_vals else 0.0

    # Optional BERTScore over aligned pairs
    bertscore_avg = 0.0
    if use_bertscore and pairs:
        a = [gold_args[gi] for gi, _, _ in pairs]
        b = [pred_args[pj] for _, pj, _ in pairs]
        bertscore_avg = berts_pairwise(a, b)

    return {
        "n_gold": len(gold_args),
        "n_pred": len(pred_args),
        "avg_pair_similarity": round(avg_sim, 4),
        "precision@thr": round(precision, 4),
        "recall@thr": round(recall, 4),
        "f1@thr": round(f1, 4),
        "rougeL_f1_avg": round(rougeL_avg, 4),
        "bleu_avg": round(bleu_avg, 4),
        **({"bertscore_f1_avg": round(bertscore_avg, 4)} if use_bertscore else {}),
    }

###################################
# Scoring by goal (precision, recall, f1)

def evaluate_by_goal(pred_by_goal, gold_by_goal, sim_threshold=0.75):
    
    goals = sorted(set(map(str, pred_by_goal.keys())) | set(map(str, gold_by_goal.keys())), key=lambda x:int(x))
    rows = []
    for g in goals:
        pred = pred_by_goal.get(g, [])
        gold = gold_by_goal.get(g, [])
        res  = evaluate_one(pred, gold, sim_threshold)
        rows.append({"goal": g, **res})
    df = pd.DataFrame(rows)

    # micro across goals
    micro = {
        "goal": "MICRO",
        "n_gold": int(df["n_gold"].sum()),
        "n_pred": int(df["n_pred"].sum()),
    }
    # recompute micro precision/recall/f1 from totals using matched counts
    # Rebuild matches to count true positives (>=thr)

    tp = 0
    for g in goals:
        pred = pred_by_goal.get(g, [])
        gold = gold_by_goal.get(g, [])
        pairs, _ = optimal_matching(gold, pred)
        tp += sum(s >= 0.75 for _, _, s in pairs)
    p = tp / max(int(df["n_pred"].sum()), 1)
    r = tp / max(int(df["n_gold"].sum()), 1)
    f1 = 2*p*r / max(p+r, 1e-9)
    micro.update({"precision@thr": round(p,4), "recall@thr": round(r,4), "f1@thr": round(f1,4)})

    return df, pd.DataFrame([micro])


In [15]:
############### Metric vs. Manual Annotation ############### 

def evaluate_models(path_args_extracted, path_gt_args, prefix, gold_model_name,
                    model_names, sim_threshold=0.75):

    def p_args(m):       return os.path.join(path_args_extracted, f"{prefix}_AllArgs_{m}_arguments.txt")
    def p_bygoal(m):     return os.path.join(path_args_extracted, f"{prefix}_AllArgs_{m}_by_goal.txt")
    def p_args_gt(m):    return os.path.join(path_gt_args,       f"{prefix}_AllArgs_{m}_arguments.txt")
    def p_bygoal_gt(m):  return os.path.join(path_gt_args,       f"{prefix}_AllArgs_{m}_by_goal.txt")

    # Embeddings
    emb_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

    def encode(texts):
        if not texts:
            return np.zeros((0, emb_model.get_sentence_embedding_dimension()), dtype=np.float32)
        return emb_model.encode(texts, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False)

    def cos_sim_mat(A, B):
        if A.size == 0 or B.size == 0:
            return np.zeros((A.shape[0], B.shape[0]), dtype=np.float32)
        return A @ B.T  # normalized -> cosine

    def greedy_match(sim, thr):
        if sim.size == 0: return []
        pairs, used_i, used_j = [], set(), set()
        idxs = np.dstack(np.unravel_index(np.argsort(sim, axis=None)[::-1], sim.shape))[0]
        for i, j in idxs:
            if sim[i, j] < thr: break
            if i in used_i or j in used_j: 
                continue
            used_i.add(int(i)); used_j.add(int(j))
            pairs.append((int(i), int(j), float(sim[i, j])))
        return pairs

    def jaccard_exact(pred, gold):
        sp, sg = set(map(normalize, pred)), set(map(normalize, gold))
        if not sp and not sg: return 1.0
        u = len(sp | sg)
        return len(sp & sg) / u if u else 0.0

    # Manual Annotations (gold)
    gold_all  = load_args_txt(p_args_gt(gold_model_name))
    gold_goal = load_by_goal_json_txt(p_bygoal_gt(gold_model_name))

    overall_rows, bygoal_frames = [], []

    #  Embeddings gold
    E_gold = encode(gold_all)
    gold_norm_set = set(map(normalize, gold_all))

    # collect predictions (strings + embeddings) for agreement
    preds_by_model = {}
    E_preds_by_model = {}

    iaa_rows = []

    for m in model_names:
        print(f"Evaluating model: {m}")
        pred_all  = load_args_txt(p_args(m))
        pred_goal = load_by_goal_json_txt(p_bygoal(m))

        # == OVERALL ==
        overall = evaluate_one(pred_all, gold_all, sim_threshold)
        E_pred_all = encode(pred_all)
        sim_all = cos_sim_mat(E_gold, E_pred_all)
        pairs_all = greedy_match(sim_all, sim_threshold)
        matched_all = len(pairs_all)
        n_gold_all = len(gold_all)
        n_pred_all = len(pred_all)
        coverage_ratio = (matched_all / n_gold_all) if n_gold_all else 0.0
        overgeneration = ((n_pred_all - matched_all) / n_pred_all) if n_pred_all else 0.0

        overall_rows.append({
            "model": m,
            **overall,
            "overgeneration": round(overgeneration, 4),
        })

        # == BY_GOAL ==
        df_by, df_micro = evaluate_by_goal(pred_goal, gold_goal, sim_threshold)
        cov_over_rows = []
        all_goals = sorted(set(map(int, gold_goal.keys())) | set(map(int, pred_goal.keys())))
        for g in all_goals:
            gold_g = gold_goal.get(str(g), []) if isinstance(next(iter(gold_goal.keys())), str) else gold_goal.get(g, [])
            pred_g = pred_goal.get(str(g), []) if isinstance(next(iter(pred_goal.keys())), str) else pred_goal.get(g, [])

            Eg = encode(gold_g)
            Ep = encode(pred_g)
            sim_g = cos_sim_mat(Eg, Ep)
            pairs_g = greedy_match(sim_g, sim_threshold)
            matched_g = len(pairs_g)
            n_gold_g = len(gold_g)
            n_pred_g = len(pred_g)
            cov_g = (matched_g / n_gold_g) if n_gold_g else 0.0
            over_g = ((n_pred_g - matched_g) / n_pred_g) if n_pred_g else 0.0
            cov_over_rows.append({"goal": g, "overgeneration": over_g})

        cov_over_df = pd.DataFrame(cov_over_rows)
        goal_col = None
        for cand in ["goal", "ods", "sdg"]:
            if cand in df_by.columns:
                goal_col = cand
                break
        if goal_col is None:
            goal_col = df_by.columns[0]

        def to_int_series(s):
            try:
                return pd.to_numeric(s, errors="raise").astype("Int64")
            except Exception:
                return None

        left_goal_int  = to_int_series(df_by[goal_col])
        right_goal_int = to_int_series(cov_over_df["goal"])

        if left_goal_int is not None and right_goal_int is not None:
            df_by = df_by.copy()
            df_by[goal_col] = left_goal_int
            cov_over_df = cov_over_df.copy()
            cov_over_df["goal"] = right_goal_int
        else:
            df_by = df_by.copy()
            df_by[goal_col] = df_by[goal_col].astype(str)
            cov_over_df = cov_over_df.copy()
            cov_over_df["goal"] = cov_over_df["goal"].astype(str)

        df_by = df_by.merge(cov_over_df.rename(columns={"goal": goal_col}),
                            on=goal_col, how="left")
        
        df_by.insert(0, "model", m)
        df_micro.insert(0, "model", m)
        bygoal_frames.append(df_by)
        bygoal_frames.append(df_micro)

        preds_by_model[m] = pred_all
        E_preds_by_model[m] = E_pred_all

        # == AGREEMENT PER MODEL ==
        jac_exact = jaccard_exact(pred_all, gold_all)
        inter = matched_all
        union = len(gold_all) + len(pred_all) - inter
        fuzzy_j = inter / union if union else 0.0

        iaa_rows.append({
            "model": m,
            "jaccard_vs_gold": round(jac_exact, 4),
            "fuzzy_jaccard_vs_gold": round(fuzzy_j, 4),
            "n_gold": len(gold_all),
            "n_pred": len(pred_all),
            "matches_at_thr": inter
        })

    iaa_per_model = pd.DataFrame(iaa_rows).sort_values("fuzzy_jaccard_vs_gold", ascending=False).reset_index(drop=True)

    # == AGREEMENT OVERALL ==
    n_gold = len(gold_all)
    counts_per_gold = np.zeros(n_gold, dtype=int)
    for m in model_names:
        E_pred = E_preds_by_model[m]
        sim = cos_sim_mat(E_gold, E_pred)
        pairs = greedy_match(sim, sim_threshold)
        hit_idxs = {gi for gi, _, _ in pairs}
        for gi in hit_idxs:
            counts_per_gold[gi] += 1

    cov_ge1 = (counts_per_gold >= 1).mean() if n_gold else 0.0
    cov_ge2 = (counts_per_gold >= 2).mean() if n_gold else 0.0
    cov_ge3 = (counts_per_gold >= 3).mean() if n_gold else 0.0
    cov_ge5 = (counts_per_gold >= 5).mean() if n_gold else 0.0

    total_preds, total_supported = 0, 0
    for m in model_names:
        E_self = E_preds_by_model[m]
        total_preds += E_self.shape[0]
        others = [E_preds_by_model[mm] for mm in model_names if mm != m and E_preds_by_model[mm].shape[0] > 0]
        if not others or E_self.shape[0] == 0:
            continue
        E_union = np.vstack(others)
        sim = cos_sim_mat(E_self, E_union)
        total_supported += int((sim.max(axis=1) >= sim_threshold).sum()) if sim.size else 0
    pct_model_supported = (total_supported / total_preds) if total_preds else 0.0

    def fleiss_kappa_binary(present_matrix: np.ndarray) -> float:
        n_items, n_raters = present_matrix.shape
        if n_items == 0 or n_raters < 2: return 0.0
        p_i = present_matrix.mean(axis=1)
        Pbar = np.mean(p_i**2 + (1 - p_i)**2)
        p_bar = present_matrix.mean()
        Pbar_e = p_bar**2 + (1 - p_bar)**2
        denom = 1.0 - Pbar_e
        return (Pbar - Pbar_e) / denom if denom > 0 else 0.0

    if n_gold and model_names:
        present = np.zeros((n_gold, len(model_names)), dtype=int)
        for j, m in enumerate(model_names):
            sim = cos_sim_mat(E_gold, E_preds_by_model[m])
            pairs = greedy_match(sim, sim_threshold)
            if pairs:
                present[list({gi for gi, _, _ in pairs}), j] = 1
        fleiss_kappa_all = float(fleiss_kappa_binary(present))
    else:
        fleiss_kappa_all = 0.0

        # --- NEW: kappa only for the big models subset ---
    big_models = {"gemma3-27b", "llama3.3-70b", "deepseek-r1-70b"}
    sel_models = [m for m in model_names if m in big_models and m in E_preds_by_model]

    if n_gold and sel_models:
        present_big = np.zeros((n_gold, len(sel_models)), dtype=int)
        for j, m in enumerate(sel_models):
            sim = cos_sim_mat(E_gold, E_preds_by_model[m])
            pairs = greedy_match(sim, sim_threshold)
            if pairs:
                present_big[list({gi for gi, _, _ in pairs}), j] = 1
        fleiss_kappa_big_models = float(fleiss_kappa_binary(present_big))
    else:
        fleiss_kappa_big_models = 0.0

    iaa_overall = pd.DataFrame([{
        "models_compared": len(model_names),
        "gold_args": n_gold,
        "coverage_ge1": round(cov_ge1, 4),
        "coverage_ge2": round(cov_ge2, 4),
        "coverage_ge3": round(cov_ge3, 4),
        "coverage_ge5": round(cov_ge5, 4),
        "pct_model_args_supported_by_another": round(pct_model_supported, 4),
        "fleiss_kappa_all": round(fleiss_kappa_all, 4),
        "fleiss_kappa_big_models": round(fleiss_kappa_big_models, 4),  # <-- NEW
    }])

    return {
        "overall": pd.DataFrame(overall_rows).sort_values("f1@thr", ascending=False).reset_index(drop=True),
        "by_goal": pd.concat(bygoal_frames, ignore_index=True),
        "iaa_per_model": iaa_per_model,
        "iaa_overall": iaa_overall
    }

# Calculate metrics by document

### Global 2023

#### GLOBAL 2023 No keywords

In [16]:
path_gt_args = "..\\Data\\Annotations"

path_args_extracted = "..\\Data\\Processed Arguments No Keywords\\Args"
path_output_metrics = "..\\Data\\Processed Arguments No Keywords\\Stats"
prefix   = "GLOBAL_SGD2023_"
gt     = "annotations"        
models   = ["qwen2.5-3b","gemma3-4b","gemma3-27b", "llama3.3-70b", "deepseek-r1-70b"]

### 2. Retrieval metrics vs. manual depuration
print("Extraction quality metrics vs. manual depuration")

results_md = compute_prf(path_output_metrics)
display(results_md)

### 1. Quality metrics vs. manual annotation
results = evaluate_models(path_args_extracted, path_gt_args,
                          prefix, gt, models, sim_threshold=0.75)

print("Extraction quality metrics vs. manual annotation")
for k, df in results.items():
    print(f"\n== {k.upper()} ==")
    display(df)

# Save the results
results["overall"].to_csv(os.path.join(path_output_metrics, f"{prefix}_ExtractionStats_overall.csv"), index=False)
results["by_goal"].to_csv(os.path.join(path_output_metrics, f"{prefix}_ExtractionStats_byGoal.csv"), index=False)
results["iaa_per_model"].to_csv(os.path.join(path_output_metrics, f"{prefix}_ExtractionStats_iaa_per_model.csv"), index=False)
results["iaa_overall"].to_csv(os.path.join(path_output_metrics, f"{prefix}_ExtractionStats_iaa_overall.csv"), index=False)

Extraction quality metrics vs. manual depuration


,document,model,total,extracted_total,real_argument,precision
0,GLOBAL SGD2023,qwen2.5-3b,397,251,161,0.6414
1,GLOBAL SGD2023,gemma3-4b,397,4093,706,0.1725
2,GLOBAL SGD2023,gemma3-27b,397,708,330,0.4661
3,GLOBAL SGD2023,llama3.3-70b,397,817,477,0.5838
4,GLOBAL SGD2023,deepseek-r1-70b,397,1178,553,0.4694
5,GLOBAL SGD2024,qwen2.5-3b,682,218,91,0.4174
6,GLOBAL SGD2024,gemma3-4b,682,3877,342,0.0882
7,GLOBAL SGD2024,gemma3-27b,682,637,317,0.4976
8,GLOBAL SGD2024,llama3.3-70b,682,836,401,0.4797
9,GLOBAL SGD2024,deepseek-r1-70b,682,1253,436,0.3480


Evaluating model: qwen2.5-3b
Evaluating model: gemma3-4b
Evaluating model: gemma3-27b
Evaluating model: llama3.3-70b
Evaluating model: deepseek-r1-70b
Extraction quality metrics vs. manual annotation

== OVERALL ==


,model,n_gold,n_pred,avg_pair_similarity,precision@thr,recall@thr,f1@thr,rougeL_f1_avg,bleu_avg,overgeneration
0,deepseek-r1-70b,235,553,0.8608,0.3038,0.7149,0.4264,0.7143,0.6229,0.6962
1,gemma3-27b,235,334,0.7463,0.3323,0.4723,0.3902,0.5200,0.4220,0.6647
2,llama3.3-70b,235,477,0.7940,0.2662,0.5404,0.3567,0.5900,0.4992,0.7338
3,gemma3-4b,235,729,0.8209,0.2003,0.6213,0.3029,0.6429,0.5349,0.7997
4,qwen2.5-3b,235,165,0.6507,0.2545,0.1787,0.2100,0.3430,0.2236,0.7394



== BY_GOAL ==


,model,goal,n_gold,n_pred,avg_pair_similarity,precision@thr,recall@thr,f1@thr,rougeL_f1_avg,bleu_avg,overgeneration
0,qwen2.5-3b,0,35,32,0.5696,0.1250,0.1143,0.1194,0.2362,0.0993,0.875000
1,qwen2.5-3b,1,18,9,0.4593,0.0000,0.0000,0.0000,0.1291,0.0306,1.000000
2,qwen2.5-3b,2,10,2,0.4726,0.0000,0.0000,0.0000,0.1374,0.0122,1.000000
3,qwen2.5-3b,3,18,12,0.4926,0.0833,0.0556,0.0667,0.1408,0.0158,0.916667
4,qwen2.5-3b,4,29,5,0.5565,0.2000,0.0345,0.0588,0.3043,0.2140,0.800000
...,...,...,...,...,...,...,...,...,...,...,...
90,deepseek-r1-70b,14,20,10,0.8082,0.6000,0.3000,0.4000,0.5628,0.4608,0.400000
91,deepseek-r1-70b,15,15,20,0.6725,0.3000,0.4000,0.3429,0.4486,0.3720,0.700000
92,deepseek-r1-70b,16,52,29,0.6195,0.2759,0.1538,0.1975,0.3569,0.2646,0.724138
93,deepseek-r1-70b,17,49,77,0.7428,0.2987,0.4694,0.3651,0.4880,0.3718,0.701299



== IAA_PER_MODEL ==


,model,jaccard_vs_gold,fuzzy_jaccard_vs_gold,n_gold,n_pred,matches_at_thr
0,deepseek-r1-70b,0.1336,0.2710,235,553,168
1,gemma3-27b,0.1278,0.2451,235,334,112
2,llama3.3-70b,0.1206,0.2171,235,477,127
3,gemma3-4b,0.0994,0.1785,235,729,146
4,qwen2.5-3b,0.0557,0.1204,235,165,43



== IAA_OVERALL ==


,models_compared,gold_args,coverage_ge1,coverage_ge2,coverage_ge3,coverage_ge5,pct_model_args_supported_by_another,fleiss_kappa_all,fleiss_kappa_big_models
0,5,235,0.9191,0.7319,0.5149,0.0809,0.8663,0.3217,0.5272


#### GLOBAL 2023 keywords

In [17]:
path_gt_args = "..\\Data\\Annotations"

path_args_extracted = "..\\Data\\Processed Arguments Keywords\\Args"
path_output_metrics = "..\\Data\\Processed Arguments Keywords\\Stats"
prefix   = "GLOBAL_SGD2023_"
gt     = "annotations"        
models   = ["llama3.3-70b","qwen2.5-3b","gemma3-4b","gemma3-27b", "deepseek-r1-70b"]

### 2. Retrieval metrics vs. manual depuration
print("Extraction quality metrics vs. manual depuration")

results_md = compute_prf(path_output_metrics)
display(results_md)

### 1. Quality metrics vs. manual annotation
results = evaluate_models(path_args_extracted, path_gt_args,
                          prefix, gt, models, sim_threshold=0.75)

print("Extraction quality metrics vs. manual annotation")
for k, df in results.items():
    print(f"\n== {k.upper()} ==")
    display(df)

# Save the results
results["overall"].to_csv(os.path.join(path_output_metrics, f"{prefix}_ExtractionStats_overall.csv"), index=False)
results["by_goal"].to_csv(os.path.join(path_output_metrics, f"{prefix}_ExtractionStats_byGoal.csv"), index=False)
results["iaa_per_model"].to_csv(os.path.join(path_output_metrics, f"{prefix}_ExtractionStats_iaa_per_model.csv"), index=False)
results["iaa_overall"].to_csv(os.path.join(path_output_metrics, f"{prefix}_ExtractionStats_iaa_overall.csv"), index=False)



Extraction quality metrics vs. manual depuration


,document,model,total,extracted_total,real_argument,precision
0,GLOBAL SGD2023,qwen2.5-3b,397,153,95,0.6209
1,GLOBAL SGD2023,gemma3-4b,397,3195,396,0.1239
2,GLOBAL SGD2023,gemma3-27b,397,697,284,0.4075
3,GLOBAL SGD2023,llama3.3-70b,397,859,359,0.4179
4,GLOBAL SGD2023,deepseek-r1-70b,397,1148,368,0.3206
5,GLOBAL SGD2024,gemma3-4b,682,4334,476,0.1098
6,GLOBAL SGD2024,gemma3-27b,682,527,363,0.6888
7,GLOBAL SGD2024,llama3.3-70b,682,862,553,0.6415
8,GLOBAL SGD2024,deepseek-r1-70b,682,1225,552,0.4506
9,GLOBAL SGD2025,qwen2.5-3b,328,51,30,0.5882


Evaluating model: llama3.3-70b
Evaluating model: qwen2.5-3b
Evaluating model: gemma3-4b
Evaluating model: gemma3-27b
Evaluating model: deepseek-r1-70b
Extraction quality metrics vs. manual annotation

== OVERALL ==


,model,n_gold,n_pred,avg_pair_similarity,precision@thr,recall@thr,f1@thr,rougeL_f1_avg,bleu_avg,overgeneration
0,deepseek-r1-70b,235,368,0.8032,0.3723,0.5830,0.4544,0.6049,0.5036,0.6304
1,gemma3-27b,235,284,0.7281,0.3873,0.4681,0.4239,0.5054,0.3954,0.6092
2,llama3.3-70b,235,359,0.7677,0.3315,0.5064,0.4007,0.5516,0.4509,0.6685
3,gemma3-4b,235,396,0.7518,0.2828,0.4766,0.3550,0.5121,0.3822,0.7146
4,qwen2.5-3b,235,96,0.6988,0.3229,0.1319,0.1873,0.3720,0.2343,0.6562



== BY_GOAL ==


,model,goal,n_gold,n_pred,avg_pair_similarity,precision@thr,recall@thr,f1@thr,rougeL_f1_avg,bleu_avg,overgeneration
0,llama3.3-70b,0,35,75,0.7303,0.1733,0.3714,0.2364,0.4484,0.3174,0.826667
1,llama3.3-70b,1,18,11,0.6378,0.2727,0.1667,0.2069,0.3966,0.2939,0.727273
2,llama3.3-70b,2,10,7,0.7085,0.4286,0.3000,0.3529,0.4853,0.4224,0.571429
3,llama3.3-70b,3,18,15,0.5922,0.3333,0.2778,0.3030,0.3439,0.2145,0.666667
4,llama3.3-70b,4,29,20,0.6381,0.4000,0.2759,0.3265,0.4293,0.2969,0.600000
...,...,...,...,...,...,...,...,...,...,...,...
90,deepseek-r1-70b,14,20,5,0.8912,0.8000,0.2000,0.3200,0.7455,0.6677,0.200000
91,deepseek-r1-70b,15,15,21,0.6367,0.1905,0.2667,0.2222,0.3614,0.2810,0.809524
92,deepseek-r1-70b,16,52,27,0.6754,0.2593,0.1346,0.1772,0.3722,0.2591,0.740741
93,deepseek-r1-70b,17,49,44,0.6969,0.3864,0.3469,0.3656,0.4347,0.3174,0.613636



== IAA_PER_MODEL ==


,model,jaccard_vs_gold,fuzzy_jaccard_vs_gold,n_gold,n_pred,matches_at_thr
0,deepseek-r1-70b,0.1211,0.2912,235,368,136
1,gemma3-27b,0.1085,0.2721,235,284,111
2,llama3.3-70b,0.1292,0.2505,235,359,119
3,gemma3-4b,0.0832,0.2181,235,396,113
4,qwen2.5-3b,0.0258,0.1107,235,96,33



== IAA_OVERALL ==


,models_compared,gold_args,coverage_ge1,coverage_ge2,coverage_ge3,coverage_ge5,pct_model_args_supported_by_another,fleiss_kappa_all,fleiss_kappa_big_models
0,5,235,0.8255,0.634,0.4298,0.0511,0.8556,0.3673,0.5947


### GLOBAL 2024

#### GLOBAL 2024 No keywords

In [18]:
path_gt_args = "..\\Data\\Annotations"

path_args_extracted = "..\\Data\\Processed Arguments No Keywords\\Args"
path_output_metrics = "..\\Data\\Processed Arguments No Keywords\\Stats"
prefix   = "GLOBAL_SGD2024_"
gt     = "annotations"        
models   = ["qwen2.5-3b","gemma3-4b","gemma3-27b", "llama3.3-70b", "deepseek-r1-70b"]

### 1. Quality metrics vs. manual annotation
results = evaluate_models(path_args_extracted, path_gt_args,
                          prefix, gt, models, sim_threshold=0.75)

print("Extraction quality metrics vs. manual annotation")
for k, df in results.items():
    print(f"\n== {k.upper()} ==")
    display(df)

# Save the results
results["overall"].to_csv(os.path.join(path_output_metrics, f"{prefix}_ExtractionStats_overall.csv"), index=False)
results["by_goal"].to_csv(os.path.join(path_output_metrics, f"{prefix}_ExtractionStats_byGoal.csv"), index=False)
results["iaa_per_model"].to_csv(os.path.join(path_output_metrics, f"{prefix}_ExtractionStats_iaa_per_model.csv"), index=False)
results["iaa_overall"].to_csv(os.path.join(path_output_metrics, f"{prefix}_ExtractionStats_iaa_overall.csv"), index=False)

Evaluating model: qwen2.5-3b
Evaluating model: gemma3-4b
Evaluating model: gemma3-27b
Evaluating model: llama3.3-70b
Evaluating model: deepseek-r1-70b
Extraction quality metrics vs. manual annotation

== OVERALL ==


,model,n_gold,n_pred,avg_pair_similarity,precision@thr,recall@thr,f1@thr,rougeL_f1_avg,bleu_avg,overgeneration
0,llama3.3-70b,415,401,0.7764,0.5786,0.5590,0.5686,0.5979,0.5315,0.4214
1,deepseek-r1-70b,415,436,0.7710,0.5252,0.5518,0.5382,0.5831,0.5134,0.4725
2,gemma3-27b,415,317,0.7891,0.5678,0.4337,0.4918,0.6131,0.5574,0.4290
3,gemma3-4b,415,342,0.7558,0.5117,0.4217,0.4624,0.5400,0.4332,0.4854
4,qwen2.5-3b,415,92,0.7918,0.5435,0.1205,0.1972,0.5505,0.4486,0.4457



== BY_GOAL ==


,model,goal,n_gold,n_pred,avg_pair_similarity,precision@thr,recall@thr,f1@thr,rougeL_f1_avg,bleu_avg,overgeneration
0,qwen2.5-3b,0,16,3,0.6437,0.3333,0.0625,0.1053,0.4389,0.3378,0.666667
1,qwen2.5-3b,1,33,14,0.5793,0.2143,0.0909,0.1277,0.2684,0.1542,0.785714
2,qwen2.5-3b,2,31,4,0.6827,0.2500,0.0323,0.0571,0.3915,0.2873,0.750000
3,qwen2.5-3b,3,56,17,0.6031,0.4118,0.1250,0.1918,0.2840,0.1641,0.588235
4,qwen2.5-3b,4,35,3,0.6504,0.3333,0.0286,0.0526,0.4335,0.3504,0.666667
...,...,...,...,...,...,...,...,...,...,...,...
90,deepseek-r1-70b,14,22,11,0.9576,0.9091,0.4545,0.6061,0.8699,0.8270,0.090909
91,deepseek-r1-70b,15,24,18,0.9067,0.8333,0.6250,0.7143,0.7930,0.7484,0.166667
92,deepseek-r1-70b,16,47,18,0.7676,0.5556,0.2128,0.3077,0.5780,0.5128,0.444444
93,deepseek-r1-70b,17,46,12,0.6329,0.2500,0.0652,0.1034,0.3220,0.2247,0.750000



== IAA_PER_MODEL ==


,model,jaccard_vs_gold,fuzzy_jaccard_vs_gold,n_gold,n_pred,matches_at_thr
0,llama3.3-70b,0.3211,0.3973,415,401,232
1,deepseek-r1-70b,0.3082,0.3704,415,436,230
2,gemma3-27b,0.3021,0.3285,415,317,181
3,gemma3-4b,0.1543,0.3029,415,342,176
4,qwen2.5-3b,0.0577,0.1118,415,92,51



== IAA_OVERALL ==


,models_compared,gold_args,coverage_ge1,coverage_ge2,coverage_ge3,coverage_ge5,pct_model_args_supported_by_another,fleiss_kappa_all,fleiss_kappa_big_models
0,5,415,0.8313,0.6361,0.412,0.0241,0.8407,0.3239,0.5218


#### GLOBAL 2024 keywords

In [19]:
path_gt_args = "..\\Data\\Annotations"

path_args_extracted = "..\\Data\\Processed Arguments Keywords\\Args"
path_output_metrics = "..\\Data\\Processed Arguments Keywords\\Stats"
prefix   = "GLOBAL_SGD2024_"
gt     = "annotations"        
models   = ["qwen2.5-3b","gemma3-4b","gemma3-27b", "llama3.3-70b", "deepseek-r1-70b"]

### 1. Quality metrics vs. manual annotation
results = evaluate_models(path_args_extracted, path_gt_args,
                          prefix, gt, models, sim_threshold=0.75)

print("Extraction quality metrics vs. manual annotation")
for k, df in results.items():
    print(f"\n== {k.upper()} ==")
    display(df)

# Save the results
results["overall"].to_csv(os.path.join(path_output_metrics, f"{prefix}_ExtractionStats_overall.csv"), index=False)
results["by_goal"].to_csv(os.path.join(path_output_metrics, f"{prefix}_ExtractionStats_byGoal.csv"), index=False)
results["iaa_per_model"].to_csv(os.path.join(path_output_metrics, f"{prefix}_ExtractionStats_iaa_per_model.csv"), index=False)
results["iaa_overall"].to_csv(os.path.join(path_output_metrics, f"{prefix}_ExtractionStats_iaa_overall.csv"), index=False)



Evaluating model: qwen2.5-3b
Evaluating model: gemma3-4b
Evaluating model: gemma3-27b
Evaluating model: llama3.3-70b
Evaluating model: deepseek-r1-70b
Extraction quality metrics vs. manual annotation

== OVERALL ==


,model,n_gold,n_pred,avg_pair_similarity,precision@thr,recall@thr,f1@thr,rougeL_f1_avg,bleu_avg,overgeneration
0,gemma3-27b,415,363,0.8025,0.6061,0.5301,0.5656,0.6373,0.5787,0.3939
1,llama3.3-70b,415,553,0.8332,0.4810,0.6410,0.5496,0.6716,0.6109,0.5172
2,deepseek-r1-70b,415,552,0.8181,0.4674,0.6217,0.5336,0.6413,0.5739,0.5326
3,gemma3-4b,415,476,0.7244,0.3971,0.4554,0.4242,0.4936,0.3900,0.6029
4,qwen2.5-3b,415,46,0.8781,0.7174,0.0795,0.1432,0.7257,0.6547,0.2826



== BY_GOAL ==


,model,goal,n_gold,n_pred,avg_pair_similarity,precision@thr,recall@thr,f1@thr,rougeL_f1_avg,bleu_avg,overgeneration
0,qwen2.5-3b,0,16,2,0.4355,0.0000,0.0000,0.0000,0.0890,0.0164,1.000000
1,qwen2.5-3b,1,33,2,0.9191,1.0000,0.0606,0.1143,0.9286,0.8407,0.000000
2,qwen2.5-3b,2,31,5,0.8275,0.8000,0.1290,0.2222,0.6667,0.5170,0.200000
3,qwen2.5-3b,3,56,2,0.8411,0.5000,0.0179,0.0345,0.5597,0.5026,0.500000
4,qwen2.5-3b,4,35,1,1.0000,1.0000,0.0286,0.0556,1.0000,1.0000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...
90,deepseek-r1-70b,14,22,19,0.8404,0.6842,0.5909,0.6341,0.7169,0.6835,0.315789
91,deepseek-r1-70b,15,24,25,0.8044,0.6000,0.6250,0.6122,0.6598,0.6262,0.400000
92,deepseek-r1-70b,16,47,19,0.7512,0.4737,0.1915,0.2727,0.5085,0.4190,0.526316
93,deepseek-r1-70b,17,46,28,0.6188,0.3214,0.1957,0.2432,0.3629,0.2717,0.678571



== IAA_PER_MODEL ==


,model,jaccard_vs_gold,fuzzy_jaccard_vs_gold,n_gold,n_pred,matches_at_thr
0,gemma3-27b,0.3526,0.3943,415,363,220
1,llama3.3-70b,0.3259,0.3809,415,553,267
2,deepseek-r1-70b,0.2965,0.3639,415,552,258
3,gemma3-4b,0.1208,0.2692,415,476,189
4,qwen2.5-3b,0.0553,0.0771,415,46,33



== IAA_OVERALL ==


,models_compared,gold_args,coverage_ge1,coverage_ge2,coverage_ge3,coverage_ge5,pct_model_args_supported_by_another,fleiss_kappa_all,fleiss_kappa_big_models
0,5,415,0.8843,0.7253,0.453,0.0313,0.8568,0.2943,0.5076


### GLOBAL 2025

#### GLOBAL 2025 keywords

In [20]:
path_gt_args = "..\\Data\\Annotations"

path_args_extracted = "..\\Data\\Processed Arguments Keywords\\Args"
path_output_metrics = "..\\Data\\Processed Arguments Keywords\\Stats"
prefix   = "GLOBAL_SGD2025_"
gt     = "annotations"        
models   = ["qwen2.5-3b","gemma3-4b","gemma3-27b", "llama3.3-70b", "deepseek-r1-70b"]

### 1. Quality metrics vs. manual annotation
results = evaluate_models(path_args_extracted, path_gt_args,
                          prefix, gt, models, sim_threshold=0.75)

print("Extraction quality metrics vs. manual annotation")
for k, df in results.items():
    print(f"\n== {k.upper()} ==")
    display(df)

# Save the results
results["overall"].to_csv(os.path.join(path_output_metrics, f"{prefix}_ExtractionStats_overall.csv"), index=False)
results["by_goal"].to_csv(os.path.join(path_output_metrics, f"{prefix}_ExtractionStats_byGoal.csv"), index=False)
results["iaa_per_model"].to_csv(os.path.join(path_output_metrics, f"{prefix}_ExtractionStats_iaa_per_model.csv"), index=False)
results["iaa_overall"].to_csv(os.path.join(path_output_metrics, f"{prefix}_ExtractionStats_iaa_overall.csv"), index=False)



Evaluating model: qwen2.5-3b
Evaluating model: gemma3-4b
Evaluating model: gemma3-27b
Evaluating model: llama3.3-70b
Evaluating model: deepseek-r1-70b
Extraction quality metrics vs. manual annotation

== OVERALL ==


,model,n_gold,n_pred,avg_pair_similarity,precision@thr,recall@thr,f1@thr,rougeL_f1_avg,bleu_avg,overgeneration
0,deepseek-r1-70b,166,202,0.7873,0.4901,0.5964,0.5380,0.6172,0.5328,0.5050
1,llama3.3-70b,166,116,0.7655,0.5345,0.3735,0.4397,0.5853,0.5070,0.4569
2,gemma3-27b,166,152,0.6843,0.4276,0.3916,0.4088,0.4632,0.3671,0.5724
3,gemma3-4b,166,217,0.7012,0.3226,0.4217,0.3655,0.4736,0.3497,0.6682
4,qwen2.5-3b,166,32,0.7747,0.5625,0.1084,0.1818,0.5606,0.4314,0.4375



== BY_GOAL ==


,model,goal,n_gold,n_pred,avg_pair_similarity,precision@thr,recall@thr,f1@thr,rougeL_f1_avg,bleu_avg,overgeneration
0,qwen2.5-3b,0,57,9,0.6516,0.2222,0.0351,0.0606,0.2651,0.1175,0.777778
1,qwen2.5-3b,1,12,4,0.5103,0.0000,0.0000,0.0000,0.2235,0.0287,1.000000
2,qwen2.5-3b,2,11,1,0.4777,0.0000,0.0000,0.0000,0.1389,0.0018,1.000000
3,qwen2.5-3b,3,11,0,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.000000
4,qwen2.5-3b,4,12,1,0.7925,1.0000,0.0833,0.1538,0.7111,0.4724,0.000000
...,...,...,...,...,...,...,...,...,...,...,...
90,deepseek-r1-70b,14,6,2,0.6774,0.5000,0.1667,0.2500,0.4738,0.2187,0.500000
91,deepseek-r1-70b,15,10,4,0.8677,0.7500,0.3000,0.4286,0.7737,0.7350,0.250000
92,deepseek-r1-70b,16,35,28,0.7287,0.5714,0.4571,0.5079,0.5858,0.4765,0.428571
93,deepseek-r1-70b,17,87,34,0.8146,0.5882,0.2299,0.3306,0.6790,0.5884,0.411765



== IAA_PER_MODEL ==


,model,jaccard_vs_gold,fuzzy_jaccard_vs_gold,n_gold,n_pred,matches_at_thr
0,deepseek-r1-70b,0.1474,0.3731,166,202,100
1,llama3.3-70b,0.1159,0.2877,166,116,63
2,gemma3-27b,0.1094,0.2569,166,152,65
3,gemma3-4b,0.0472,0.2315,166,217,72
4,qwen2.5-3b,0.0215,0.1000,166,32,18



== IAA_OVERALL ==


,models_compared,gold_args,coverage_ge1,coverage_ge2,coverage_ge3,coverage_ge5,pct_model_args_supported_by_another,fleiss_kappa_all,fleiss_kappa_big_models
0,5,166,0.8133,0.5783,0.3554,0.0181,0.7497,0.3169,0.5092


### OCDE 2024


#### OCDE 2024 keywords

In [21]:
path_gt_args = "..\\Data\\Annotations"

path_args_extracted = "..\\Data\\Processed Arguments Keywords\\Args"
path_output_metrics = "..\\Data\\Processed Arguments Keywords\\Stats"
prefix   = "OCDE_2024_"
gt     = "annotations"        
models   = ["qwen2.5-3b","gemma3-4b","gemma3-27b", "llama3.3-70b", "deepseek-r1-70b"]

### 1. Quality metrics vs. manual annotation
results = evaluate_models(path_args_extracted, path_gt_args,
                          prefix, gt, models, sim_threshold=0.75)

print("Extraction quality metrics vs. manual annotation")
for k, df in results.items():
    print(f"\n== {k.upper()} ==")
    display(df)

# Save the results
results["overall"].to_csv(os.path.join(path_output_metrics, f"{prefix}_ExtractionStats_overall.csv"), index=False)
results["by_goal"].to_csv(os.path.join(path_output_metrics, f"{prefix}_ExtractionStats_byGoal.csv"), index=False)
results["iaa_per_model"].to_csv(os.path.join(path_output_metrics, f"{prefix}_ExtractionStats_iaa_per_model.csv"), index=False)
results["iaa_overall"].to_csv(os.path.join(path_output_metrics, f"{prefix}_ExtractionStats_iaa_overall.csv"), index=False)



Evaluating model: qwen2.5-3b
Evaluating model: gemma3-4b
Evaluating model: gemma3-27b
Evaluating model: llama3.3-70b
Evaluating model: deepseek-r1-70b
Extraction quality metrics vs. manual annotation

== OVERALL ==


,model,n_gold,n_pred,avg_pair_similarity,precision@thr,recall@thr,f1@thr,rougeL_f1_avg,bleu_avg,overgeneration
0,deepseek-r1-70b,150,230,0.8765,0.4913,0.7533,0.5947,0.7400,0.6425,0.5043
1,llama3.3-70b,150,245,0.8645,0.4408,0.7200,0.5468,0.7079,0.6081,0.5592
2,gemma3-27b,150,205,0.8152,0.4585,0.6267,0.5296,0.6189,0.4961,0.5317
3,gemma3-4b,150,199,0.7802,0.3920,0.5200,0.4470,0.5614,0.4090,0.6080
4,qwen2.5-3b,150,52,0.7198,0.3846,0.1333,0.1980,0.4701,0.3004,0.6154



== BY_GOAL ==


,model,goal,n_gold,n_pred,avg_pair_similarity,precision@thr,recall@thr,f1@thr,rougeL_f1_avg,bleu_avg,overgeneration
0,qwen2.5-3b,0,15,5,0.6166,0.2000,0.0667,0.1000,0.3053,0.1469,0.800000
1,qwen2.5-3b,1,23,4,0.5191,0.0000,0.0000,0.0000,0.1025,0.0171,1.000000
2,qwen2.5-3b,2,40,7,0.7004,0.1429,0.0250,0.0426,0.3099,0.1925,0.857143
3,qwen2.5-3b,3,10,3,0.4885,0.0000,0.0000,0.0000,0.2528,0.0506,1.000000
4,qwen2.5-3b,4,7,2,0.5155,0.0000,0.0000,0.0000,0.1942,0.0420,1.000000
...,...,...,...,...,...,...,...,...,...,...,...
90,deepseek-r1-70b,14,2,0,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.000000
91,deepseek-r1-70b,15,2,1,0.4004,0.0000,0.0000,0.0000,0.1667,0.0393,1.000000
92,deepseek-r1-70b,16,26,2,0.6810,0.5000,0.0385,0.0714,0.5477,0.3822,0.500000
93,deepseek-r1-70b,17,26,23,0.7276,0.4348,0.3846,0.4082,0.4449,0.3200,0.521739



== IAA_PER_MODEL ==


,model,jaccard_vs_gold,fuzzy_jaccard_vs_gold,n_gold,n_pred,matches_at_thr
0,deepseek-r1-70b,0.1683,0.4286,150,230,114
1,llama3.3-70b,0.1794,0.3763,150,245,108
2,gemma3-27b,0.1348,0.3707,150,205,96
3,gemma3-4b,0.0850,0.2878,150,199,78
4,qwen2.5-3b,0.0256,0.1099,150,52,20



== IAA_OVERALL ==


,models_compared,gold_args,coverage_ge1,coverage_ge2,coverage_ge3,coverage_ge5,pct_model_args_supported_by_another,fleiss_kappa_all,fleiss_kappa_big_models
0,5,150,0.8933,0.7933,0.6267,0.08,0.9066,0.3393,0.5926


### G20 2024

#### G20 2024 keywords

In [23]:
path_gt_args = "..\\Data\\Annotations"

path_args_extracted = "..\\Data\\Processed Arguments Keywords\\Args"
path_output_metrics = "..\\Data\\Processed Arguments Keywords\\Stats"
prefix   = "G20_2024_"
gt     = "annotations"        
models   = ["qwen2.5-3b","gemma3-4b","gemma3-27b", "llama3.3-70b", "deepseek-r1-70b"]

### 2. Retrieval metrics vs. manual depuration
print("Extraction quality metrics vs. manual depuration")

results_md = compute_prf(path_output_metrics)
display(results_md)

### 1. Quality metrics vs. manual annotation
results = evaluate_models(path_args_extracted, path_gt_args,
                          prefix, gt, models, sim_threshold=0.75)

print("Extraction quality metrics vs. manual annotation")
for k, df in results.items():
    print(f"\n== {k.upper()} ==")
    display(df)

# Save the results
results["overall"].to_csv(os.path.join(path_output_metrics, f"{prefix}_ExtractionStats_overall.csv"), index=False)
results["by_goal"].to_csv(os.path.join(path_output_metrics, f"{prefix}_ExtractionStats_byGoal.csv"), index=False)
results["iaa_per_model"].to_csv(os.path.join(path_output_metrics, f"{prefix}_ExtractionStats_iaa_per_model.csv"), index=False)
results["iaa_overall"].to_csv(os.path.join(path_output_metrics, f"{prefix}_ExtractionStats_iaa_overall.csv"), index=False)



Extraction quality metrics vs. manual depuration


,document,model,total,extracted_total,real_argument,precision
0,GLOBAL SGD2023,qwen2.5-3b,397,153,95,0.6209
1,GLOBAL SGD2023,gemma3-4b,397,3195,396,0.1239
2,GLOBAL SGD2023,gemma3-27b,397,697,284,0.4075
3,GLOBAL SGD2023,llama3.3-70b,397,859,359,0.4179
4,GLOBAL SGD2023,deepseek-r1-70b,397,1148,368,0.3206
5,GLOBAL SGD2024,gemma3-4b,682,4334,476,0.1098
6,GLOBAL SGD2024,gemma3-27b,682,527,363,0.6888
7,GLOBAL SGD2024,llama3.3-70b,682,862,553,0.6415
8,GLOBAL SGD2024,deepseek-r1-70b,682,1225,552,0.4506
9,GLOBAL SGD2025,qwen2.5-3b,328,51,30,0.5882


Evaluating model: qwen2.5-3b
Evaluating model: gemma3-4b
Evaluating model: gemma3-27b
Evaluating model: llama3.3-70b
Evaluating model: deepseek-r1-70b
Extraction quality metrics vs. manual annotation

== OVERALL ==


,model,n_gold,n_pred,avg_pair_similarity,precision@thr,recall@thr,f1@thr,rougeL_f1_avg,bleu_avg,overgeneration
0,deepseek-r1-70b,50,46,0.8166,0.6304,0.58,0.6042,0.6421,0.5426,0.3696
1,llama3.3-70b,50,44,0.8131,0.5909,0.52,0.5532,0.6514,0.5710,0.3636
2,gemma3-27b,50,41,0.7958,0.6098,0.50,0.5495,0.6036,0.5107,0.3659
3,gemma3-4b,50,42,0.8099,0.5952,0.50,0.5435,0.5773,0.4537,0.4286
4,qwen2.5-3b,50,7,0.8361,0.5714,0.08,0.1404,0.5994,0.4469,0.4286



== BY_GOAL ==


,model,goal,n_gold,n_pred,avg_pair_similarity,precision@thr,recall@thr,f1@thr,rougeL_f1_avg,bleu_avg,overgeneration
0,qwen2.5-3b,0,11,3,0.8073,0.3333,0.0909,0.1429,0.4509,0.3662,0.666667
1,qwen2.5-3b,1,2,0,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.000000
2,qwen2.5-3b,2,0,0,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.000000
3,qwen2.5-3b,3,2,0,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.000000
4,qwen2.5-3b,4,0,0,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...
90,deepseek-r1-70b,14,1,0,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.000000
91,deepseek-r1-70b,15,5,2,0.6940,0.5000,0.2000,0.2857,0.4745,0.3422,0.500000
92,deepseek-r1-70b,16,5,1,0.4488,0.0000,0.0000,0.0000,0.1000,0.0152,1.000000
93,deepseek-r1-70b,17,32,21,0.8750,0.7143,0.4688,0.5660,0.7475,0.6578,0.285714



== IAA_PER_MODEL ==


,model,jaccard_vs_gold,fuzzy_jaccard_vs_gold,n_gold,n_pred,matches_at_thr
0,deepseek-r1-70b,0.1892,0.4328,50,46,29
1,llama3.3-70b,0.1733,0.4242,50,44,28
2,gemma3-27b,0.1351,0.4000,50,41,26
3,gemma3-4b,0.1410,0.3529,50,42,24
4,qwen2.5-3b,0.0182,0.0755,50,7,4



== IAA_OVERALL ==


,models_compared,gold_args,coverage_ge1,coverage_ge2,coverage_ge3,coverage_ge5,pct_model_args_supported_by_another,fleiss_kappa_all,fleiss_kappa_big_models
0,5,50,0.86,0.62,0.44,0.02,0.8333,0.3454,0.5684
